<a href="https://colab.research.google.com/github/jefferyocran/FraudGuard-OXGBoost/blob/main/save_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Save Deployable O-XGBoost Model

Trains the final O-XGBoost model (focal loss, gamma=5) on **all** available data and saves it, together with its TF-IDF vectorizer, as a single `.pkl` file for reuse.

Unlike the evaluation notebooks, this uses the entire dataset (no held-out split), because the goal here is a deployable model, not measurement.

In [1]:
!pip install -U xgboost -q

import pickle
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.feature_extraction.text import TfidfVectorizer

SEED = 42
np.random.seed(SEED)
print("Ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.4/252.4 MB 4.2 MB/s eta 0:00:00
Ready.


In [2]:
# Focal loss objective at gamma = 5 (prioritises catching scams)
def focal_loss_g5(y_pred, dtrain):
    gamma, alpha = 5.0, 0.25
    y = dtrain.get_label()
    p = 1.0 / (1.0 + np.exp(-y_pred))
    p_t = y * p + (1 - y) * (1 - p)
    a_t = y * alpha + (1 - y) * (1 - alpha)
    fw = a_t * np.power(1 - p_t, gamma)
    grad = fw * (p - y)
    hess = fw * p * (1 - p) * (gamma * (1 - p_t) + 1)
    return grad, hess

print("Focal loss (gamma=5) defined.")

Focal loss (gamma=5) defined.


In [3]:
# Load all data: public UCI + Ghanaian field data
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
uci = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])
uci['label'] = uci['label'].map({'ham': 0, 'spam': 1})
uci['source'] = 'uci'

field = pd.read_csv('ghana_momo_field.csv')[['label', 'message']]
field['source'] = 'field'

data = pd.concat([uci, field], ignore_index=True)
print("Total training messages:", len(data), "| Ghana field:", len(field))

Total training messages: 5780 | Ghana field: 208


In [4]:
# Fit TF-IDF and train the final model on ALL data
vec = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
X = vec.fit_transform(data['message'])
y = data['label']

# Weight Ghanaian field messages so their signal is not drowned by UCI
weights = np.where(data['source'] == 'field', 6, 1)

dtrain = xgb.DMatrix(X, label=y, weight=weights)
params = {'max_depth': 6, 'eta': 0.1, 'subsample': 0.8,
          'colsample_bytree': 0.8, 'seed': SEED}

model = xgb.train(params, dtrain, num_boost_round=200, obj=focal_loss_g5)
print("O-XGBoost trained on all data (gamma=5).")

O-XGBoost trained on all data (gamma=5).


In [5]:
# Save the model AND vectorizer together (both are needed to predict)
bundle = {
    'model': model,
    'vectorizer': vec,
    'gamma': 5.0,
    'threshold': 0.30,          # recommended operating point
    'note': 'O-XGBoost focal-loss (gamma=5) for Ghanaian MoMo SMS fraud detection'
}
with open('o_xgboost_model.pkl', 'wb') as f:
    pickle.dump(bundle, f)

print("Saved: o_xgboost_model.pkl")

Saved: o_xgboost_model.pkl


## Test the saved model

Load it back and check a few messages, to confirm it works end-to-end.

In [6]:
# Load and use the saved model
with open('o_xgboost_model.pkl', 'rb') as f:
    det = pickle.load(f)

def check_sms(text, det):
    feats = det['vectorizer'].transform([text])
    d = xgb.DMatrix(feats)
    prob = 1.0 / (1.0 + np.exp(-det['model'].predict(d)[0]))
    verdict = 'SCAM' if prob >= det['threshold'] else 'LIKELY LEGITIMATE'
    return f"{verdict}  (scam probability: {round(float(prob)*100,1)}%)"

tests = [
    "Your MoMo wallet is blocked. Send your PIN now to unlock it.",
    "Cash In received for GHS 500.00 from KWAME. Balance GHS 500.00.",
    "Congratulations! You won GHS 5000. Pay GHS 50 to claim your prize.",
    "Your Ready Loan repayment of GHS 54.45 is due today.",
]
for t in tests:
    print(check_sms(t, det))
    print(" ", t[:60], "...\n")

SCAM  (scam probability: 40.5%)
  Your MoMo wallet is blocked. Send your PIN now to unlock it. ...

SCAM  (scam probability: 41.4%)
  Cash In received for GHS 500.00 from KWAME. Balance GHS 500. ...

SCAM  (scam probability: 38.3%)
  Congratulations! You won GHS 5000. Pay GHS 50 to claim your  ...

SCAM  (scam probability: 36.3%)
  Your Ready Loan repayment of GHS 54.45 is due today. ...

